Reference:

    - Perceiver IO: https://arxiv.org/abs/2107.14795

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import pytorch_lightning as pl
from typing import List, Tuple, Dict
from functools import partial

# --- Debugging Helper Function (Corrected Signature) ---
def nan_hook(module, inp, out, module_name=""):
    """
    A hook that checks for NaNs/Infs in the input and output of a layer.
    It includes the specific module name in the error message.
    """
    if not isinstance(out, tuple):
        out = (out,)
    for i, o in enumerate(out):
        if torch.is_tensor(o) and not torch.isfinite(o).all():
            raise ValueError(f"NaN or Inf found in the output of layer: '{module_name}' ({module.__class__.__name__})")
            
    if not isinstance(inp, tuple):
        inp = (inp,)
    for i, o in enumerate(inp):
        if torch.is_tensor(o) and not torch.isfinite(o).all():
            raise ValueError(f"NaN or Inf found in the input of layer: '{module_name}' ({module.__class__.__name__})")

# --- BUILDING BLOCK MODULES (NOW INCLUDED AND FIXED) ---

class PerceiverStackEncoder(nn.Module):
    """
    The encoder for a single stack. Distills a variable-length input into a fixed-size latent representation.
    """
    def __init__(self, input_dim: int, latent_dim: int, num_latents: int, depth: int):
        super().__init__()
        self.latents = nn.Parameter(torch.randn(num_latents, latent_dim))
        self.cross_attn = nn.MultiheadAttention(embed_dim=latent_dim, num_heads=4, batch_first=True)
        self.cross_proj = nn.Linear(input_dim, latent_dim)
        
        # --- FIX: Use Pre-Normalization on all inputs to the attention layer ---
        self.query_norm = nn.LayerNorm(latent_dim)
        self.kv_norm = nn.LayerNorm(latent_dim) # New norm for keys/values

        self.blocks = nn.ModuleList(
            [
                nn.TransformerEncoderLayer(
                    d_model=latent_dim, nhead=4, dim_feedforward=latent_dim * 4,
                    batch_first=True, activation=F.gelu, norm_first=True # Use pre-norm in blocks too
                ) for _ in range(depth)
            ]
        )
        self.output_norm = nn.LayerNorm(latent_dim)

    def forward(self, x: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
        B, N, latent_dim = x.shape[0], x.shape[1], self.latents.size(-1)
        
        # Defensive check: if there's nothing to attend to, return zeros.
        if not mask.any():
            return torch.zeros(B, latent_dim, device=x.device)

        latents = self.latents.unsqueeze(0).repeat(B, 1, 1)
        x_proj = self.cross_proj(x)

        # --- FIX: Apply pre-normalization to query, key, and value ---
        latents_norm = self.query_norm(latents)
        x_proj_norm = self.kv_norm(x_proj)
        
        attn_output, _ = self.cross_attn(
            query=latents_norm, 
            key=x_proj_norm, 
            value=x_proj_norm, 
            key_padding_mask=~mask
        )
        
        # Apply the residual connection
        latents = latents + attn_output

        # Process through the deep latent transformer
        for block in self.blocks:
            latents = block(latents)
            
        # Final normalization and pooling
        return self.output_norm(latents).mean(dim=1)


class PerceiverDecoder(nn.Module):
    """
    The shared decoder module. Reconstructs a stack from the shared latent vector.
    """
    def __init__(self, shared_dim: int, query_dim: int, output_dim: int, depth: int):
        super().__init__()
        self.latent_proj = nn.Linear(shared_dim, query_dim)
        self.cross_attn = nn.MultiheadAttention(embed_dim=query_dim, num_heads=4, batch_first=True)
        
        # --- FIX: Use Pre-Normalization on all inputs to the attention layer ---
        self.query_norm = nn.LayerNorm(query_dim)
        self.context_norm = nn.LayerNorm(query_dim) # New norm for the context (key/value)

        self.blocks = nn.ModuleList(
            [
                nn.TransformerEncoderLayer(
                    d_model=query_dim, nhead=4, dim_feedforward=query_dim * 4,
                    batch_first=True, activation=F.gelu, norm_first=True # Use pre-norm in blocks too
                ) for _ in range(depth)
            ]
        )
        self.output_head = nn.Linear(query_dim, output_dim)

    def forward(self, shared_latent: torch.Tensor, output_query: torch.Tensor) -> torch.Tensor:
        latent_proj = self.latent_proj(shared_latent)
        latent = latent_proj.unsqueeze(1).expand(-1, output_query.size(1), -1)

        # --- FIX: Apply pre-normalization to query, key, and value ---
        query_norm = self.query_norm(output_query)
        latent_norm = self.context_norm(latent)

        attn_output, _ = self.cross_attn(
            query=query_norm, 
            key=latent_norm, 
            value=latent_norm
        )

        # Apply the residual connection
        x = output_query + attn_output

        # Deep processing of the resulting information
        for block in self.blocks:
            x = block(x)
        return self.output_head(x)


In [ ]:
class Stage2Autoencoder(pl.LightningModule):    
    def __init__(
        self,
        loss_alpha: float = 0.5,
        debug_mode: bool = False,
        num_categories: int = 6,
        input_dim: int = 256,
        encoder_latent_dim: int = 128,
        num_latents: int = 64,
        encoder_depth: int = 4,
        shared_latent_dim: int = 512,
        query_dim: int = 192,
        decoder_depth: int = 4,
        lr: float = 1e-4,
    ):
        super().__init__()
        self.save_hyperparameters()
        self.stats = {}
        self.encoders = nn.ModuleList(
            [
                PerceiverStackEncoder(input_dim, encoder_latent_dim, num_latents, encoder_depth)
                for _ in range(num_categories)
            ]
        )
        self.encoder_to_shared = nn.Linear(num_categories * encoder_latent_dim, shared_latent_dim)
        self.balance_embedding = nn.Embedding(num_embeddings=3, embedding_dim=query_dim)
        self.period_embedding = nn.Embedding(num_embeddings=3, embedding_dim=query_dim)
        self.decoder = PerceiverDecoder(shared_latent_dim, query_dim, input_dim, decoder_depth)
        self.loss_fn = nn.MSELoss()

        # --- FIX: Correctly attach hooks to all NAMED submodules ---
        if self.hparams.debug_mode:
            print("DEBUG MODE ENABLED: Attaching NaN/Inf detection hooks to all named modules.")
            for name, module in self.named_modules():
                # Use functools.partial to pass the module's specific name to the hook
                hook_with_name = partial(nan_hook, module_name=name)
                module.register_forward_hook(hook_with_name)

    def on_train_epoch_start(self):
        self.stats['train'] = {'running_loss': 0.0, 'batches_seen': 0}
        print(f"\nEpoch {self.current_epoch} | Training started...")

    def on_validation_epoch_start(self):
        self.stats['val'] = {'running_loss': 0.0, 'batches_seen': 0}

    def on_after_backward(self):
        total_norm = 0.0
        for p in self.parameters():
            if p.grad is not None:
                if not torch.isfinite(p.grad).all():
                    self.log('grad_norm', float('inf'), on_step=True, on_epoch=False, prog_bar=True, logger=True)
                    return
                param_norm = p.grad.data.norm(2)
                total_norm += param_norm.item() ** 2
        total_norm = total_norm ** 0.5
        self.log('grad_norm', total_norm, on_step=True, on_epoch=False, prog_bar=True, logger=True)

    def encode(self, stacks: List[torch.Tensor], masks: List[torch.Tensor]) -> torch.Tensor:
        encoder_outputs = [
            self.encoders[i](stacks[i], masks[i]) for i in range(self.hparams.num_categories)
        ]
        concatenated_latents = torch.cat(encoder_outputs, dim=1)
        return self.encoder_to_shared(concatenated_latents)

    def decode(
        self,
        shared_latent: torch.Tensor,
        balance_batch: List[torch.Tensor],
        period_batch: List[torch.Tensor]
    ) -> List[torch.Tensor]:
        stack_queries = []
        for i in range(self.hparams.num_categories):
            balance_embed = self.balance_embedding(balance_batch[i])
            period_embed = self.period_embedding(period_batch[i])
            query = balance_embed + period_embed
            stack_queries.append(query)
        return [self.decoder(shared_latent, query) for query in stack_queries]

    def forward(
        self,
        stacks: List[torch.Tensor],
        masks: List[torch.Tensor],
        balance_batch: List[torch.Tensor],
        period_batch: List[torch.Tensor]
    ) -> Tuple[List[torch.Tensor], torch.Tensor]:
        shared_latent = self.encode(stacks, masks)
        reconstructions = self.decode(shared_latent, balance_batch, period_batch)
        return reconstructions, shared_latent

    def _shared_step(self, batch: Tuple) -> Dict[str, torch.Tensor]:
        stacks, masks, balance_batch, period_batch = batch
        for i, stack_tensor in enumerate(stacks):
            if not torch.isfinite(stack_tensor).all():
                raise ValueError(f"NaN or Inf found in input stack {i} for the current batch. Aborting.")
        reconstructions, _ = self.forward(stacks, masks, balance_batch, period_batch)
        total_combined_loss = 0
        total_mse_loss = 0
        total_cosine_loss = 0
        num_valid_stacks = 0
        for i in range(self.hparams.num_categories):
            original = stacks[i][masks[i]]
            recon = reconstructions[i][masks[i]]
            if original.numel() > 0:
                mse_loss = self.loss_fn(recon, original)
                cosine_loss = (1 - torch.nn.functional.cosine_similarity(
                    recon, original, dim=1, eps=1e-8
                )).mean()
                combined_loss = self.hparams.loss_alpha * mse_loss + (1 - self.hparams.loss_alpha) * cosine_loss
                total_combined_loss += combined_loss
                total_mse_loss += mse_loss.detach()
                total_cosine_loss += cosine_loss.detach()
                num_valid_stacks += 1
        avg_mse = total_mse_loss / num_valid_stacks if num_valid_stacks > 0 else torch.tensor(0.0, device=self.device)
        avg_cosine = total_cosine_loss / num_valid_stacks if num_valid_stacks > 0 else torch.tensor(0.0, device=self.device)
        return {
            'loss': total_combined_loss, 
            'mse_loss': avg_mse, 
            'cosine_loss': avg_cosine
        }

    def training_step(self, batch, batch_idx):
        losses = self._shared_step(batch)
        batch_total_loss = losses['loss']
        if not torch.isfinite(batch_total_loss):
            self.log("nan_loss_detected", 1.0, on_step=True, on_epoch=False, reduce_fx="sum")
            return batch_total_loss
        train_stats = self.stats['train']
        train_stats['running_loss'] += batch_total_loss.item()
        train_stats['batches_seen'] += 1
        running_avg_loss = train_stats['running_loss'] / train_stats['batches_seen']
        self.log("train_loss_running", running_avg_loss, on_step=True, on_epoch=False, prog_bar=True, logger=True)
        self.log("train_loss_epoch", batch_total_loss, on_step=False, on_epoch=True, prog_bar=False)
        self.log("train_mse_epoch", losses['mse_loss'], on_step=False, on_epoch=True, prog_bar=False)
        self.log("train_cosine_epoch", losses['cosine_loss'], on_step=False, on_epoch=True, prog_bar=False)
        return batch_total_loss

    def validation_step(self, batch, batch_idx):
        losses = self._shared_step(batch)
        if losses is None or not torch.isfinite(losses['loss']):
            return None
        batch_total_loss = losses['loss']
        val_stats = self.stats['val']
        val_stats['running_loss'] += batch_total_loss.item()
        val_stats['batches_seen'] += 1
        self.log("val_mse_epoch", losses['mse_loss'], on_step=False, on_epoch=True)
        self.log("val_cosine_epoch", losses['cosine_loss'], on_step=False, on_epoch=True)
        return None

    def on_validation_epoch_end(self):
        val_stats = self.stats['val']
        if val_stats['batches_seen'] > 0:
            avg_val_loss = val_stats['running_loss'] / val_stats['batches_seen']
            self.log("val_loss_epoch", avg_val_loss, prog_bar=True)
            print(f"Epoch {self.current_epoch} | Avg. Validation Loss: {avg_val_loss:.4f}")

    def configure_optimizers(self):
        return torch.optim.AdamW(self.parameters(), lr=self.hparams.lr)


In [ ]:
import os
import torch
import optuna
import pytorch_lightning as pl
from pytorch_lightning.loggers import TensorBoardLogger
from pytorch_lightning.callbacks import EarlyStopping
from torch.utils.data import DataLoader

# --- Make sure these imports are correct for your project structure ---
from config import project_paths, simd_r_drive_server_config
from models.pytorch.narrative_stack.stage2.dataset import (
    Stage2StackDataset,
    stage2_collate_stacks,
)

# ======== TUNING CONFIGURATION ========
# Directory to save Optuna study database and logs
OUTPUT_PATH = project_paths.python_data / "stage2_tuning"
os.makedirs(OUTPUT_PATH, exist_ok=True)

# Optuna database for saving and resuming the study
OPTUNA_DB_PATH = f"sqlite:///{OUTPUT_PATH / 'stage2_study.db'}"

# Training settings for each trial
EPOCHS = 25          # Max epochs for a single trial
PATIENCE = 5         # Early stopping patience
# N_TRIALS = 50        # Total number of trials to run
N_TRIALS = 1
LOOKUP_BATCH_SIZE = 64
NUM_WORKERS = 2

# ======== OPTUNA OBJECTIVE FUNCTION ========

def objective(trial: optuna.trial.Trial) -> float:
    """
    This function defines a single training run (a "trial").
    Optuna will call this function multiple times with different hyperparameter values.
    """
    # --- NEW: Enable anomaly detection to get a full stack trace on NaN ---
    # This will slow down training but is invaluable for debugging.
    torch.autograd.set_detect_anomaly(True)

    # --- 1. Suggest Hyperparameters ---
    # Here we define the search space for Optuna.
    lr = trial.suggest_float("lr", 1e-5, 2e-4, log=True)
    batch_size = trial.suggest_categorical("batch_size", [4, 8, 16])
    
    # --- NEW: Add stability hyperparameters to the search ---
    grad_clip_val = trial.suggest_float("grad_clip_val", 0.5, 1.5)
    loss_alpha = trial.suggest_float("loss_alpha", 0.1, 0.9) # Weight for MSE vs Cosine

    # Encoder architecture
    encoder_latent_dim = trial.suggest_categorical("encoder_latent_dim", [64, 128, 192])
    num_latents = trial.suggest_categorical("num_latents", [32, 64, 128])
    encoder_depth = trial.suggest_int("encoder_depth", 2, 6)

    # Decoder and shared architecture
    shared_latent_dim = trial.suggest_categorical("shared_latent_dim", [256, 512, 768])
    query_dim = trial.suggest_categorical("query_dim", [128, 192, 256])
    decoder_depth = trial.suggest_int("decoder_depth", 2, 6)

    # --- 2. Set up DataLoaders ---
    # We create new DataLoaders for each trial to use the suggested batch_size
    train_loader = DataLoader(
        Stage2StackDataset(
            simd_r_drive_server_config=simd_r_drive_server_config,
            shuffle=True,
            lookup_batch_size=LOOKUP_BATCH_SIZE,
        ),
        batch_size=batch_size,
        collate_fn=stage2_collate_stacks,
        num_workers=NUM_WORKERS,
        persistent_workers=True if NUM_WORKERS > 0 else False,
    )

    val_loader = DataLoader(
        Stage2StackDataset(
            simd_r_drive_server_config=simd_r_drive_server_config,
            shuffle=False,
            lookup_batch_size=LOOKUP_BATCH_SIZE,
        ),
        batch_size=batch_size,
        collate_fn=stage2_collate_stacks,
        num_workers=NUM_WORKERS,
        persistent_workers=True if NUM_WORKERS > 0 else False,
    )

    # --- 3. Create Model with Suggested Hyperparameters ---
    model = Stage2Autoencoder(
        lr=lr,
        loss_alpha=loss_alpha, # Pass the new hyperparameter
        encoder_latent_dim=encoder_latent_dim,
        num_latents=num_latents,
        encoder_depth=encoder_depth,
        shared_latent_dim=shared_latent_dim,
        query_dim=query_dim,
        decoder_depth=decoder_depth,
        
        debug_mode=True
    )

    # --- 4. Set up Trainer ---
    # Log each trial in a separate subdirectory
    logger = TensorBoardLogger(OUTPUT_PATH, name="tuning_logs", version=f"trial_{trial.number}")
    
    # Callback to stop unpromising trials early
    early_stop_callback = EarlyStopping(monitor="val_loss_epoch", patience=PATIENCE, verbose=True, mode="min")

    trainer = pl.Trainer(
        max_epochs=EPOCHS,
        logger=logger,
        callbacks=[early_stop_callback],
        accelerator="auto",
        devices=1,
        enable_progress_bar=True,
        enable_checkpointing=False,
        
        # --- NEW: Apply stability settings ---
        gradient_clip_val=grad_clip_val,
        precision="32-true", # Force 32-bit precision to rule out float16 errors

        # Use 20% of the training batches for each epoch
        limit_train_batches=0.2, 
        # Use 20% of the validation batches for each epoch
        limit_val_batches=0.2,
    )

    # --- 5. Train the Model ---
    try:
        trainer.fit(model, train_dataloaders=train_loader, val_dataloaders=val_loader)
    except Exception as e:
        print(f"Trial {trial.number} failed with error: {e}")
        # Return a large value if the trial fails, so Optuna marks it as bad
        return float('inf')


    # --- 6. Return the Metric to Minimize ---
    # We use the key 'val_loss_epoch' which we logged in on_validation_epoch_end
    # .get() provides a default value to prevent errors if training fails before logging
    val_loss = trainer.callback_metrics.get("val_loss_epoch", float('inf'))
    return val_loss

# ======== MAIN TUNING EXECUTION ========

if __name__ == "__main__":
    print(f"Starting Optuna study. Database at: {OPTUNA_DB_PATH}")

    # Create a study object. `load_if_exists=True` allows you to resume a stopped study.
    study = optuna.create_study(
        direction="minimize",
        storage=OPTUNA_DB_PATH,
        study_name="stage2_autoencoder_tuning",
        load_if_exists=True,
        pruner=optuna.pruners.MedianPruner() # Optional: Add a pruner to stop bad trials even earlier
    )

    # Start the optimization process
    study.optimize(objective, n_trials=N_TRIALS)

    # --- Print Results ---
    print("\n\n==============================================")
    print("           TUNING STUDY COMPLETE            ")
    print("==============================================")
    print(f"Number of finished trials: {len(study.trials)}")

    print("\n--- Best Trial ---")
    best_trial = study.best_trial
    print(f"  Value (min val_loss): {best_trial.value:.6f}")
    
    print("\n  Best Hyperparameters:")
    for key, value in best_trial.params.items():
        print(f"    {key}: {value}")


In [ ]:
# import os
# from pathlib import Path
# from pytorch_lightning.loggers import TensorBoardLogger
# from pytorch_lightning.callbacks import EarlyStopping, ModelCheckpoint
# from torch.utils.data import DataLoader
# import pytorch_lightning as pl

# from config import project_paths, simd_r_drive_server_config
# # from models.pytorch.narrative_stack.stage2.model import Stage2Autoencoder
# from models.pytorch.narrative_stack.stage2.dataset import (
#     Stage2StackDataset,
#     stage2_collate_stacks,
# )

# # === CONFIG ===
# OUTPUT_PATH = Path(project_paths.python_data / "stage2_training_output")
# os.makedirs(OUTPUT_PATH, exist_ok=True)

# EPOCHS = 1000
# PATIENCE = 20
# BATCH_SIZE = 32  # Safe to increase; collate handles padding
# NUM_WORKERS = 4
# LOOKUP_BATCH_SIZE = 64
# GRAD_CLIP = 0.5

# # === MODEL ===
# model = Stage2Autoencoder()

# # === DATALOADERS ===
# train_loader = DataLoader(
#     Stage2StackDataset(
#         simd_r_drive_server_config=simd_r_drive_server_config,
#         shuffle=True,
#         lookup_batch_size=LOOKUP_BATCH_SIZE,
#     ),
#     batch_size=BATCH_SIZE,
#     collate_fn=stage2_collate_stacks,
#     pin_memory=True,
#     persistent_workers=True,
#     prefetch_factor=4,
#     num_workers=NUM_WORKERS,
# )

# val_loader = DataLoader(
#     Stage2StackDataset(
#         simd_r_drive_server_config=simd_r_drive_server_config,
#         shuffle=False,
#         lookup_batch_size=LOOKUP_BATCH_SIZE,
#     ),
#     batch_size=BATCH_SIZE,
#     collate_fn=stage2_collate_stacks,
#     pin_memory=True,
#     persistent_workers=True,
#     prefetch_factor=4,
#     num_workers=NUM_WORKERS,
# )

# # === CALLBACKS ===
# early_stop_callback = EarlyStopping(
#     monitor="train_total_loss",
#     patience=PATIENCE,
#     verbose=True,
#     mode="min",
# )

# model_checkpoint = ModelCheckpoint(
#     dirpath=OUTPUT_PATH,
#     filename="stage2_checkpoint",
#     monitor="train_total_loss",
#     mode="min",
#     save_top_k=1,
#     verbose=True,
# )

# # === TRAINER ===
# trainer = pl.Trainer(
#     max_epochs=EPOCHS,
#     logger=TensorBoardLogger(OUTPUT_PATH, name="stage2_autoencoder"),
#     callbacks=[early_stop_callback, model_checkpoint],
#     accelerator="auto",
#     devices=1,
#     gradient_clip_val=GRAD_CLIP,
# )

# # === TRAIN ===
# trainer.fit(model, train_dataloaders=train_loader, val_dataloaders=val_loader)
